# Bayesian BAI-MCTS Experiments

Colab notebook for testing the recursive Bayesian MCTS ideas in NanoZero:

- logit-shifted Gaussian priors
- Bayesian optimality weights
- prior/search consensus stopping
- epsilon-tie stopping
- IDS allocation ablations (`precision` vs `visits`)
- final policy ablations (`optimality` vs `consensus`)

Recommended runtime: CPU is enough for TicTacToe calibration; use a GPU for larger Connect4/model runs.

## 1. Setup

When opened through Colab's GitHub integration, this notebook still clones the repository into `/content/nanozero` so imports and Rust builds are reproducible. Change `BRANCH` if you want to test a feature branch.

In [ ]:
REPO_URL = "https://github.com/caldred/nanozero.git"
BRANCH = "main"

!rm -rf /content/nanozero
!git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/nanozero
%cd /content/nanozero

!pip install -q numpy scipy matplotlib maturin

# Colab CPU runtimes do not always have Cargo on PATH.
import os
if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    !curl https://sh.rustup.rs -sSf | sh -s -- -y --profile minimal
os.environ["PATH"] = f"{os.environ['HOME']}/.cargo/bin:" + os.environ["PATH"]
!cargo --version

%cd /content/nanozero/nanozero-mcts-rs
!maturin build --release
!pip install -q target/wheels/nanozero_mcts_rs-*.whl --force-reinstall
%cd /content/nanozero

!python -c "from nanozero_mcts_rs import RustBatchedMCTS, RustBayesianMCTS; print('Rust MCTS backends loaded')"

In [ ]:
import math
import subprocess
import sys
import time
import numpy as np
import torch
import matplotlib.pyplot as plt

from nanozero.common import set_seed
from nanozero.config import BayesianMCTSConfig, MCTSConfig, get_model_config
from nanozero.game import get_game
from nanozero.mcts import BayesianMCTS, BatchedMCTS
from nanozero.model import AlphaZeroTransformer

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

class UniformModel(torch.nn.Module):
    def __init__(self, action_size):
        super().__init__()
        self.action_size = action_size
        self.dummy = torch.nn.Parameter(torch.zeros(1))

    def predict(self, x, action_mask=None):
        mask = action_mask.float()
        probs = mask / mask.sum(dim=-1, keepdim=True).clamp_min(1.0)
        values = torch.zeros((x.shape[0], 1), device=x.device)
        return probs, values

def entropy(p, eps=1e-12):
    p = np.asarray(p, dtype=np.float64)
    p = p[p > 0]
    return float(-(p * np.log(p + eps)).sum())

def summarize_policy(game, policy, state=None, top_k=5):
    state = game.initial_state() if state is None else state
    legal = game.legal_actions(state)
    ranked = sorted([(a, float(policy[a])) for a in legal], key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

## 2. Search Sanity Check

Runs PUCT and Bayesian MCTS on initial TicTacToe and Connect4 states with a uniform model. This verifies the Rust extension, Python wrapper, policy normalization, and search diagnostics.

Expected result: these empty-board checks should be nearly uniform. They are plumbing tests, not evidence that Bayesian search has found a preference. The tactical check below uses a position where search should break symmetry.

In [ ]:
def run_search_sanity(game_name, sims=64):
    game = get_game(game_name)
    model = UniformModel(game.config.action_size).to(device).eval()
    state = game.initial_state()[np.newaxis, ...]

    puct = BatchedMCTS(
        game,
        MCTSConfig(num_simulations=sims, c_puct=1.0),
        leaves_per_batch=1,
        use_transposition_table=False,
    )
    bayes = BayesianMCTS(
        game,
        BayesianMCTSConfig(num_simulations=sims, early_stopping=False),
        leaves_per_batch=1,
        seed=42,
        use_transposition_table=False,
    )

    puct_policy = puct.search(state, model, add_noise=False)[0]
    bayes_policy = bayes.search(state, model)[0]
    stats = bayes.search_stats()[0]

    print(f"\n{game_name}")
    print("PUCT sum:", round(float(puct_policy.sum()), 6), "entropy:", round(entropy(puct_policy), 4))
    print("Bayes sum:", round(float(bayes_policy.sum()), 6), "entropy:", round(entropy(bayes_policy), 4))
    print("Bayes stats:", stats)
    print("Bayes top actions:", summarize_policy(game, bayes_policy, state))

run_search_sanity("tictactoe")
run_search_sanity("connect4")

## 2b. Tactical TicTacToe Check

This position has one important move: X must block action `5`, otherwise O wins immediately. A uniform model is intentionally used so the result comes from tree search and terminal backups, not from policy bias.

In [ ]:
def run_tactical_tictactoe(sims=200):
    game = get_game("tictactoe")
    model = UniformModel(game.config.action_size).to(device).eval()
    state = game.initial_state()
    for action in [0, 3, 6, 4]:
        state = game.next_state(state, action)

    puct = BatchedMCTS(
        game,
        MCTSConfig(num_simulations=sims, c_puct=1.0),
        leaves_per_batch=1,
        use_transposition_table=False,
    )
    bayes = BayesianMCTS(
        game,
        BayesianMCTSConfig(num_simulations=sims, early_stopping=False),
        leaves_per_batch=1,
        seed=42,
        use_transposition_table=False,
    )

    puct_policy = puct.search(state[np.newaxis, ...], model, add_noise=False)[0]
    bayes_policy = bayes.search(state[np.newaxis, ...], model)[0]
    stats = bayes.search_stats()[0]

    print(game.display(state))
    print("Legal actions:", game.legal_actions(state))
    print("Expected block action: 5")
    print("PUCT top actions:", summarize_policy(game, puct_policy, state))
    print("Bayes top actions:", summarize_policy(game, bayes_policy, state))
    print("Bayes stats:", stats)

run_tactical_tictactoe()

## 3. Bayesian Ablations

Compare the two new knobs:

- `ids_allocation`: `precision` or `visits`
- `final_policy`: `optimality` or `consensus`

The output focuses on search cost, stop reason, consensus score, tie gap, and policy entropy.

In [ ]:
def bayesian_ablation(game_name="connect4", sims=128, early_stopping=True):
    game = get_game(game_name)
    model = UniformModel(game.config.action_size).to(device).eval()
    state = game.initial_state()[np.newaxis, ...]
    rows = []

    for ids_allocation in ["precision", "visits"]:
        for final_policy in ["optimality", "consensus"]:
            cfg = BayesianMCTSConfig(
                num_simulations=sims,
                sigma_0=1.0,
                obs_var=0.5,
                ids_alpha=0.5,
                ids_allocation=ids_allocation,
                final_policy=final_policy,
                early_stopping=early_stopping,
                confidence_threshold=0.99,
                epsilon_tie=0.02,
                tie_sigma=1.0,
            )
            mcts = BayesianMCTS(game, cfg, leaves_per_batch=1, seed=7, use_transposition_table=False)
            t0 = time.perf_counter()
            policy = mcts.search(state, model)[0]
            elapsed = time.perf_counter() - t0
            stats = mcts.search_stats()[0]
            rows.append({
                "ids": ids_allocation,
                "policy": final_policy,
                "sims": stats["simulations_used"],
                "stop": stats["stop_reason"],
                "consensus": stats["consensus_score"],
                "tie_gap": stats["tie_gap"],
                "entropy": entropy(policy),
                "top_action": int(np.argmax(policy)),
                "seconds": elapsed,
            })
    return rows

for game_name in ["tictactoe", "connect4"]:
    print("\n", game_name.upper())
    for row in bayesian_ablation(game_name, sims=128, early_stopping=True):
        print(row)

## 4. Calibration Against Exact TicTacToe

This runs the repo's calibration script. It samples TicTacToe positions, computes exact minimax action values, and compares Bayesian consensus confidence to actual best-action correctness.

Use the ECE and bucket table to tune `sigma_0`, `obs_var`, `confidence_threshold`, `epsilon_tie`, and the ablation knobs.

In [ ]:
!python scripts/calibrate_bayesian.py \
  --positions 200 \
  --simulations 100 \
  --buckets 10 \
  --ids_alpha 0.5 \
  --ids_allocation precision \
  --final_policy optimality \
  --sigma_0 1.0 \
  --obs_var 0.5 \
  --confidence_threshold 0.99 \
  --epsilon_tie 0.02 \
  --tie_sigma 1.0 \
  --seed 42

## 5. Small Calibration Grid

A cheap grid over the two ablation knobs. For serious runs, increase `--positions` and `--simulations`.

In [ ]:
for ids in ["precision", "visits"]:
    for pol in ["optimality", "consensus"]:
        print("\n===", ids, pol, "===")
        result = subprocess.run([
            sys.executable,
            "scripts/calibrate_bayesian.py",
            "--positions", "80",
            "--simulations", "64",
            "--buckets", "8",
            "--ids_allocation", ids,
            "--final_policy", pol,
            "--ids_alpha", "0.5",
            "--seed", "123",
        ], check=False, capture_output=True, text=True)
        if result.stdout:
            print(result.stdout.strip())
        if result.stderr:
            print(result.stderr.strip())
        if result.returncode != 0:
            raise RuntimeError(f"calibration grid failed for ids={ids}, policy={pol}")

## 6. Optional Root-Bandit Benchmark With a Checkpoint

If you have a trained checkpoint, upload it to Colab and set `CHECKPOINT`. This compares UCB-style root selection against TTTS-IDS with the new stop metrics.

In [ ]:
CHECKPOINT = None  # e.g. "/content/connect4_iter100.pt"

if CHECKPOINT:
    subprocess.run([
        sys.executable,
        "scripts/benchmark_root_bandits.py",
        "--game", "connect4",
        "--checkpoint", CHECKPOINT,
        "--n_pulls", "25", "50", "100",
        "--n_trials", "30",
        "--n_positions", "10",
        "--ids_allocation", "precision",
        "--final_policy", "optimality",
    ], check=True)
else:
    print("Set CHECKPOINT to run the root-bandit benchmark.")